In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt



In [3]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
print(len(words))

32033


In [5]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [6]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
  
  print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)


emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [7]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [8]:
# 我们有27个字符，假设每个字符有个二维的特征空间
# 构建look-up table
C = torch.randn((27, 2))

In [9]:
C

tensor([[-0.3544,  0.3958],
        [-1.9757, -1.7096],
        [-0.4497, -0.5883],
        [ 0.3690,  0.9676],
        [ 0.1688,  1.6989],
        [ 1.1933, -0.1064],
        [-0.4947,  0.4072],
        [-0.0724,  0.1619],
        [ 0.5884, -0.7265],
        [-0.2547, -1.4808],
        [-0.8332,  1.7538],
        [-0.9700,  0.0868],
        [-0.7441, -0.9968],
        [-0.5174, -0.8554],
        [ 1.2076, -0.3410],
        [-1.5700,  0.4911],
        [ 0.2411, -0.9650],
        [-1.9836, -1.6881],
        [ 0.3175, -0.5609],
        [-1.1827, -0.0353],
        [ 0.6617, -1.5950],
        [ 1.2584, -0.0157],
        [-0.4719, -0.3447],
        [ 0.6061, -2.0472],
        [-0.1829, -1.2031],
        [ 2.1814, -0.2707],
        [ 0.6848,  0.9305]])

In [10]:
# 计算输入5的嵌入向量
C[5]


tensor([ 1.1933, -0.1064])

In [12]:
F.one_hot(torch.tensor(5), num_classes=27).float()@C

tensor([ 1.1933, -0.1064])

In [13]:
# 计算X的嵌入向量
C[X]

tensor([[[-0.3544,  0.3958],
         [-0.3544,  0.3958],
         [-0.3544,  0.3958]],

        [[-0.3544,  0.3958],
         [-0.3544,  0.3958],
         [ 1.1933, -0.1064]],

        [[-0.3544,  0.3958],
         [ 1.1933, -0.1064],
         [-0.5174, -0.8554]],

        [[ 1.1933, -0.1064],
         [-0.5174, -0.8554],
         [-0.5174, -0.8554]],

        [[-0.5174, -0.8554],
         [-0.5174, -0.8554],
         [-1.9757, -1.7096]],

        [[-0.3544,  0.3958],
         [-0.3544,  0.3958],
         [-0.3544,  0.3958]],

        [[-0.3544,  0.3958],
         [-0.3544,  0.3958],
         [-1.5700,  0.4911]],

        [[-0.3544,  0.3958],
         [-1.5700,  0.4911],
         [-0.7441, -0.9968]],

        [[-1.5700,  0.4911],
         [-0.7441, -0.9968],
         [-0.2547, -1.4808]],

        [[-0.7441, -0.9968],
         [-0.2547, -1.4808],
         [-0.4719, -0.3447]],

        [[-0.2547, -1.4808],
         [-0.4719, -0.3447],
         [-0.2547, -1.4808]],

        [[-0.4719, -0

In [14]:
C[X].shape

torch.Size([32, 3, 2])

In [15]:
# 对于输入的32*3的X的每一个整数，计算了特征检索向量-2维的特征
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [18]:
# 每次输入的是3个字符，每个是2维度的，假设我们此处有100个神经元
W1 = torch.randn(6, 100)
b1 = torch.rand(100) # 100个偏置

# emb@W1 + b1
# (32 3 2) @ (6,100) 
h = emb.view(32, 6) @ W1 + b1
h

tensor([[ 0.3805,  0.1159,  0.3366,  ...,  0.9164, -0.3914,  1.9612],
        [-0.8540,  1.1486, -0.1463,  ...,  0.2634, -0.2785,  3.2748],
        [-0.0233, -1.5962,  1.8384,  ...,  0.2074, -0.3486,  0.8271],
        ...,
        [ 2.0516,  0.6776,  1.7698,  ..., -0.4977, -1.6846,  3.9114],
        [ 0.4490, -0.7219, -0.2690,  ..., -0.4228,  1.2794, -0.9743],
        [ 1.1645, -2.2480, -1.0920,  ..., -0.3279, -0.3150, -4.7282]])

In [17]:
h.shape

torch.Size([32, 100])

In [19]:
# 移除硬编码
# h = emb.view(emb.shape[0], 6) @ W1 + b1
# or 
h = emb.view(-1, 6) @ W1 + b1
h

tensor([[ 0.3805,  0.1159,  0.3366,  ...,  0.9164, -0.3914,  1.9612],
        [-0.8540,  1.1486, -0.1463,  ...,  0.2634, -0.2785,  3.2748],
        [-0.0233, -1.5962,  1.8384,  ...,  0.2074, -0.3486,  0.8271],
        ...,
        [ 2.0516,  0.6776,  1.7698,  ..., -0.4977, -1.6846,  3.9114],
        [ 0.4490, -0.7219, -0.2690,  ..., -0.4228,  1.2794, -0.9743],
        [ 1.1645, -2.2480, -1.0920,  ..., -0.3279, -0.3150, -4.7282]])

In [20]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
h

tensor([[ 0.3632,  0.1154,  0.3244,  ...,  0.7242, -0.3726,  0.9612],
        [-0.6931,  0.8173, -0.1453,  ...,  0.2575, -0.2715,  0.9971],
        [-0.0233, -0.9211,  0.9506,  ...,  0.2045, -0.3351,  0.6789],
        ...,
        [ 0.9675,  0.5899,  0.9436,  ..., -0.4603, -0.9335,  0.9992],
        [ 0.4211, -0.6181, -0.2627,  ..., -0.3993,  0.8563, -0.7506],
        [ 0.8225, -0.9779, -0.7976,  ..., -0.3167, -0.3049, -0.9998]])

In [ ]:
# emb.view(-1, 6) @ W1 --- (32, 100)
# b1 --- 100

# 32 100
#  1 100
# 从右边开始，维度以啊莫一样，要么是1，广播可以正确运行


In [21]:
# h输出是32*100

W2 = torch.randn(100, 27) # 100个神经元，一共27个可能字符的概率分布
b2 = torch.rand(27) # 27个偏置



In [22]:
logits = h@W2+b2

In [23]:
logits.shape

torch.Size([32, 27])

In [25]:
counts = logits.exp()
prob = counts/counts.sum(1, keepdims=True)

In [26]:
prob.shape

torch.Size([32, 27])

In [27]:
prob[0].sum()

tensor(1.)

In [28]:
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(21.3593)

In [ ]:
# ------------ now made respectable :) ---------------

In [32]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [33]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [34]:
emb = C[X] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
logits = h@W2+b2

counts = logits.exp()
prob = counts/counts.sum(1, keepdims=True)
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(17.7697)

In [35]:
F.cross_entropy(logits, Y)

tensor(17.7697)

In [ ]:
loss = F.cross_entropy(logits, Y)

In [36]:
for p in parameters:
    p.requires_grad = True


In [42]:
for _ in range(1000):
    # forward pass
    emb = C[X] # (32, 3, 2)
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
    logits = h@W2+b2
    loss = F.cross_entropy(logits, Y)
    # print(loss.item())

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -0.1 * p.grad
print(loss.item())

0.2552211880683899


In [43]:
logits.max(1)

torch.return_types.max(
values=tensor([13.4871, 18.1044, 20.7258, 20.8151, 16.9594, 13.4871, 16.1917, 14.3670,
        16.0963, 18.6382, 16.1848, 21.1582, 13.4871, 17.3894, 17.3903, 20.3210,
        13.4871, 16.6965, 15.4115, 17.3172, 18.7858, 16.2183, 11.0970, 10.8827,
        15.6575, 13.4871, 16.4268, 17.1676, 12.8921, 16.3777, 19.3421, 16.3297],
       grad_fn=<MaxBackward0>),
indices=tensor([ 9, 13, 13,  1,  0,  9, 12,  9, 22,  9,  1,  0,  9, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0,  9, 15, 16,  8,  9,  1,  0]))

In [40]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [ ]:
# 同预测标签和接近，但是这里loss下降不到0，因为...的下一个标签可能是e/o/i某一个

In [62]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words:
  
  # print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    # print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

In [63]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [64]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [67]:
for p in parameters:
    p.requires_grad = True


In [77]:
for _ in range(1000):

    # mini batch
    ix = torch.randint(0, X.shape[0], (32,)) # 一批次32个，随机采样
    # forward pass
    emb = C[X[ix]] # (32, 3, 2)
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
    logits = h@W2+b2
    loss = F.cross_entropy(logits, Y[ix])
    # print(loss.item())

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -0.1 * p.grad
print(loss.item())

2.266544818878174


In [78]:
# loss at whole data
emb = C[X] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
logits = h@W2+b2
loss = F.cross_entropy(logits, Y)
loss.item()

2.553175687789917

In [ ]:
# train, val ,test
# 80, 10, 10
# 总数据的80%，也就是训练数据用来训练参数，10%用来验证集，调整超参数，10%测试，测试最终的效果
